# FinNews DistilBERT Distillation — Kaggle GPU Training

Upload `labels.jsonl` as a Kaggle dataset. This notebook fine-tunes DistilBERT on GPU.

After training, download `model/` and `model.onnx` to `ml/news_classifier/` in your OpenAlgo repo.

In [ ]:
!pip install -q transformers datasets torch scikit-learn

In [ ]:
import json, numpy as np
from pathlib import Path

DATA_PATH = '/kaggle/input/finnews-labels/labels.jsonl'
OUTPUT_DIR = '/kaggle/working/model'
ONNX_PATH = '/kaggle/working/model.onnx'
EPOCHS = 5
MAX_LEN = 128

In [ ]:
texts, labels = [], []
with open(DATA_PATH) as f:
    for line in f:
        row = json.loads(line)
        texts.append(row['text'])
        labels.append(int(row['label']))
print(f'Loaded {len(texts)} examples')

In [ ]:
from transformers import DistilBertTokenizerFast, DistilBertForSequenceClassification, Trainer, TrainingArguments
from datasets import Dataset
from sklearn.metrics import accuracy_score, f1_score

tokenizer = DistilBertTokenizerFast.from_pretrained('distilbert-base-uncased')
model = DistilBertForSequenceClassification.from_pretrained('distilbert-base-uncased', num_labels=3)

enc = tokenizer(texts, truncation=True, padding=True, max_length=MAX_LEN)
dataset = Dataset.from_dict({'input_ids': enc['input_ids'], 'attention_mask': enc['attention_mask'], 'labels': labels})
split = dataset.train_test_split(test_size=0.15, seed=42)

def compute_metrics(pred):
    logits, labs = pred
    preds = np.argmax(logits, axis=-1)
    return {'accuracy': accuracy_score(labs, preds), 'f1': f1_score(labs, preds, average='weighted')}

args = TrainingArguments(OUTPUT_DIR, num_train_epochs=EPOCHS, per_device_train_batch_size=16,
    eval_strategy='epoch', save_strategy='epoch', load_best_model_at_end=True,
    metric_for_best_model='f1', logging_steps=20, report_to='none')
trainer = Trainer(model=model, args=args, train_dataset=split['train'], eval_dataset=split['test'], compute_metrics=compute_metrics)
trainer.train()
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print('Training complete')

In [ ]:
import torch
model_eval = DistilBertForSequenceClassification.from_pretrained(OUTPUT_DIR)
model_eval.eval()
dummy = tokenizer('Nifty hits all-time high', return_tensors='pt', padding='max_length', max_length=MAX_LEN, truncation=True)
torch.onnx.export(model_eval, (dummy['input_ids'], dummy['attention_mask']), ONNX_PATH,
    input_names=['input_ids','attention_mask'], output_names=['logits'],
    dynamic_axes={'input_ids':{0:'batch',1:'seq'},'attention_mask':{0:'batch',1:'seq'}}, opset_version=14)
print(f'ONNX exported to {ONNX_PATH}')
print('Download model/ and model.onnx from /kaggle/working/')